# Exp 38: Ensemble plateau thresholds

Card: [exp38_ensemble_plateau_thresholds.md](../../experiments/exp38_ensemble_plateau_thresholds.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import CVResult, Config
from core.scores import Scores

SMOKE = os.environ.get("SMOKE") == "1"
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
MODELS = {
    "exp26 ResNet-18": Config.from_toml(
        CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.toml"
    ),
    "EfficientNet-B0": Config.from_toml(
        CONFIG_DIR / "exp32_efficientnet_b0_5fold" / "efficientnet_b0.toml"
    ),
    "RegNetY-016": Config.from_toml(
        CONFIG_DIR / "exp33_regnety_016" / "regnety_016.toml"
    ),
    "ConvNeXt-Tiny": Config.from_toml(
        CONFIG_DIR / "exp34_convnext_tiny" / "convnext_tiny.toml"
    ),
}
if SMOKE:
    MODELS = dict(list(MODELS.items())[:2])
MARGIN = 0.05  # plateau: F1 >= max - MARGIN
PROBS = [f"p_{c}" for c in CLASSES]
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp38_ensemble_plateau_thresholds"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
INK, PEAK, MIDDLE = "#52514e", "#eb6834", "#2a78d6"
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": INK,
        "xtick.color": INK,
        "ytick.color": INK,
        "axes.titlesize": "medium",
        "axes.titlecolor": INK,
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)

## Ensemble


In [ ]:
oofs = [CVResult.load(c).oof.sort_index() for c in MODELS.values()]
assert all(
    o.index.equals(oofs[0].index) and o["fold"].equals(oofs[0]["fold"]) for o in oofs
)
# mean of out-of-fold probs per class; labels and folds shared by all members
oof = oofs[0][["fold", *CLASSES]].join(sum(o[PROBS] for o in oofs) / len(oofs))
labels, probs = oof[CLASSES].to_numpy(), oof[PROBS].to_numpy()

## F1 against threshold


In [ ]:
peak, plateau = Scores(labels, probs).best_f1(), Scores(labels, probs).plateau_f1(MARGIN)
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.6 * len(CLASSES), 3.6), sharey=True
)
for c, (cls, ax) in enumerate(zip(CLASSES, axes)):
    precision, recall, thr = precision_recall_curve(labels[:, c], probs[:, c])
    f1 = (2 * precision * recall / (precision + recall + 1e-12))[:-1]
    ax.axvspan(
        plateau.loc[cls, "low"],
        plateau.loc[cls, "high"],
        color=MIDDLE,
        alpha=0.12,
        lw=0,
        label=f"F1 within {MARGIN} of max",
    )
    ax.plot(thr, f1, color=INK, lw=1.5)
    best_t, mean_t = peak.loc[cls, "threshold"], plateau.loc[cls, "threshold"]
    ax.axvline(best_t, color=PEAK, lw=2, ls="--", label="best_t (peak)")
    ax.axvline(mean_t, color=MIDDLE, lw=2, label="mean_t (plateau middle)")
    # value labels at two heights, outward from the pair so they never cross a line; both left near right edge
    for t, label, y in [(best_t, "best_t", 0.22), (mean_t, "mean_t", 0.08)]:
        right = t == max(best_t, mean_t) and t < 0.8
        ax.text(
            t + 0.015 if right else min(best_t, mean_t) - 0.015,
            y,
            f"{label} {t:.3f}",
            ha="left" if right else "right",
            fontsize=8,
            color=INK,
        )
    ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
    ax.set(
        title=cls,
        xlabel="threshold",
        xlim=(0, 1),
        ylim=(0, 1.05),
        xticks=ticks,
        yticks=ticks,
    )
    ax.tick_params(labelleft=True, labelsize=8)
axes[0].set_ylabel("ensemble\nF1")
fig.suptitle(
    f"F1 against threshold, mean of {len(MODELS)} models, out-of-fold rows of folds 0–4",
    color=INK,
)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncols=3,
    bbox_to_anchor=(0.5, 0.93),
)
figures = {f"{PREFIX}_f1_curves.png": fig}
plt.show()

## Peak vs plateau thresholds


In [ ]:
RULES = {"peak": best_f1, "plateau": lambda y, p: Scores(y, p).plateau_f1(MARGIN)}


def honest_macro_f1(oof, rule):
    """Macro F1 on pooled OOF rows, each fold scored with thresholds of `rule` tuned on other folds."""
    pred = pd.DataFrame(0, index=oof.index, columns=CLASSES)
    for fold in oof["fold"].unique():
        rest, part = oof[oof["fold"] != fold], oof["fold"] == fold
        thresholds = rule(rest[CLASSES].to_numpy(), rest[PROBS].to_numpy())["threshold"]
        pred.loc[part] = (
            oof.loc[part, PROBS].to_numpy() >= thresholds.to_numpy()
        ).astype(int)
    return f1_score(oof[CLASSES], pred, average="macro", zero_division=0)


table = pd.concat(
    {rule: f(labels, probs)[["threshold", "f1"]] for rule, f in RULES.items()}, axis=1
)
honest = pd.Series(
    {rule: honest_macro_f1(oof, f) for rule, f in RULES.items()}, name="honest Macro F1"
)
display(table.round(3), honest.round(3))

In [ ]:
if SAVE_FIGURES:
    for fname, fig in figures.items():
        fig.savefig(FIGURES / fname, dpi=120)